In [ ]:
#setup
import math
import statistics as st
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

np.set_printoptions(precision=4, suppress=True)
pd.options.display.precision = 4
print(sys.executable)

In [ ]:
#data types in pandas
df = pd.DataFrame({
    "city": ["Delhi", "Pune", "Delhi", "Mumbai"],            # nominal
    "rating": pd.Categorical(["good", "poor", "good", "average"],
                             categories=["poor", "average", "good"], ordered=True),   # ordinal
    "children": [2, 0, 1, 3],                                # discrete
    "height_cm": [162.5, 175.0, 168.2, 180.1],               # continuous
})
print(df.dtypes)
print(df["rating"].max(), "| median rating position:", df["rating"].cat.codes.median())
df

In [ ]:
#one population, many possible samples
rng = np.random.default_rng(42)                                   # a seeded generator: repeatable results
population = rng.normal(loc=170, scale=10, size=100_000).round(1)  # heights of 100,000 people
mu = population.mean()
print(f"Population mean (a parameter, mu): {mu:.3f}")

for n in (5, 30, 500):
    sample = rng.choice(population, size=n, replace=False)
    print(f"n = {n:>3}: sample mean (a statistic) = {sample.mean():.3f}   error = {sample.mean() - mu:+.3f}")

In [ ]:
#the sampling distribution of the mean
means_30 = np.array([rng.choice(population, size=30).mean() for _ in range(2000)])
means_300 = np.array([rng.choice(population, size=300).mean() for _ in range(2000)])

print(f"n=30 : mean of sample means {means_30.mean():.2f}, spread {means_30.std():.2f}")
print(f"n=300: mean of sample means {means_300.mean():.2f}, spread {means_300.std():.2f}")

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(means_30, bins=40, alpha=0.6, label="n = 30")
ax.hist(means_300, bins=40, alpha=0.6, label="n = 300")
ax.axvline(mu, color="black", lw=2, label="population mean")
ax.set_xlabel("sample mean (cm)")
ax.legend()
plt.show()

In [ ]:
#a large biased sample is still wrong
cutoff = np.percentile(population, 80)
tall_only = population[population > cutoff]               # e.g. a survey run at a basketball court

random_sample = rng.choice(population, size=5000)
biased_sample = rng.choice(tall_only, size=5000)
print(f"population mean      : {mu:.2f}")
print(f"random sample (5000) : {random_sample.mean():.2f}")
print(f"biased sample (5000) : {biased_sample.mean():.2f}")

In [ ]:
#simple random vs stratified sampling
people = pd.DataFrame({
    "city": rng.choice(["Delhi", "Pune", "Mumbai"], size=1000, p=[0.6, 0.3, 0.1]),
    "income": rng.lognormal(10.5, 0.5, 1000).round(),
})
print("population shares:\n", people["city"].value_counts(normalize=True).round(3), "\n")

simple = people.sample(n=100, random_state=1)
strat = people.groupby("city", group_keys=False).sample(frac=0.10, random_state=1)
print("simple random shares:\n", simple["city"].value_counts(normalize=True).round(3), "\n")
print("stratified shares:\n", strat["city"].value_counts(normalize=True).round(3))

In [ ]:
#the arithmetic mean
x = np.array([4, 8, 6, 5, 3, 8, 9, 7, 8, 5])

print(x.sum() / len(x), x.mean(), st.mean(x), pd.Series(x).mean())      # four ways, one answer

print("sum of deviations from the mean:", round((x - x.mean()).sum(), 10))   # always 0

candidates = np.linspace(3, 9, 601)                                     # which centre fits best?
squared_error = [((x - c) ** 2).sum() for c in candidates]
print("value minimising the squared error:", candidates[np.argmin(squared_error)])

In [ ]:
#weighted mean and a frequency table
marks = [80, 90, 70]
weights = [0.3, 0.5, 0.2]                                   # assignment, exam, project
print("weighted mean:", np.average(marks, weights=weights), "| plain mean:", np.mean(marks))

values = np.array([1, 2, 3, 4])                             # a frequency table: how many times each appeared
freq = np.array([5, 10, 20, 5])
print("mean from a frequency table:", (values * freq).sum() / freq.sum(), np.average(values, weights=freq))

In [ ]:
#the median
print(st.median([3, 1, 2]), st.median([5, 1, 3, 2]), np.median([5, 1, 3, 2]))
print(sorted([5, 1, 3, 2]))                                 # 1, 2, 3, 5: the middle pair is 2 and 3

short = np.array([1, 2, 3, 4, 5])
spoiled = np.array([1, 2, 3, 4, 500])
print("mean   :", short.mean(), "->", spoiled.mean())
print("median :", np.median(short), "->", np.median(spoiled))        # barely moves

In [ ]:
#the mode
shirts = ["M", "L", "M", "S", "M", "L", "L", "XL"]
print(st.mode(shirts), st.multimode(shirts))                # a tie: mode returns the first, multimode returns all

print(pd.Series([1, 2, 2, 3, 3]).mode().tolist())           # pandas returns every mode
print(st.multimode([1, 2, 3]))                              # all values tie, so there is no useful mode

heights = population[:2000]                                 # continuous data: almost no value repeats...
counts, edges = np.histogram(heights, bins=20)              # ...so we find the modal BIN instead
i = counts.argmax()
print(f"modal bin: {edges[i]:.1f} to {edges[i + 1]:.1f} ({counts[i]} people)")

In [ ]:
#the harmonic mean
# Drive 60 km at 40 km/h, then 60 km at 60 km/h. What was the average speed?
time_taken = 60 / 40 + 60 / 60                              # 1.5 h + 1.0 h = 2.5 h
print("true average speed:", 120 / time_taken)              # 48.0 km/h
print("arithmetic mean   :", st.mean([40, 60]))             # 50: wrong
print("harmonic mean     :", st.harmonic_mean([40, 60]))    # 48: right

# Equal money (Rs 600) on apples at Rs 40/kg and oranges at Rs 60/kg: average price per kg?
kg = 600 / 40 + 600 / 60                                    # 15 kg + 10 kg
print("average price per kg:", 1200 / kg, st.harmonic_mean([40, 60]))

# It needs positive values
try:
    st.harmonic_mean([-1, 2])
except st.StatisticsError as e:
    print("StatisticsError:", e)
print("a zero gives:", st.harmonic_mean([0, 5, 10]))

In [ ]:
#the geometric mean
# Returns of +10%, -20%, +30% in three years
factors = np.array([1.10, 0.80, 1.30])
total = factors.prod()
g = st.geometric_mean(factors)
print(f"total growth factor: {total:.4f}")
print(f"arithmetic mean of the returns: {np.mean([10, -20, 30]):.2f}% a year   (misleading)")
print(f"geometric mean growth: {(g - 1) * 100:.2f}% a year")
print(f"check: 1000 * g**3 = {1000 * g ** 3:.2f}   and   1000 * total = {1000 * total:.2f}")

# +50% then -50% looks like a 0% average, but you are down 25%
print(1.5 * 0.5, st.geometric_mean([1.5, 0.5]))

# Three equivalent calculations
print(stats.gmean(factors), np.exp(np.log(factors).mean()), st.geometric_mean(factors))

for bad in ([0, 4], [-1, 4]):
    try:
        print(st.geometric_mean(bad))
    except st.StatisticsError as e:
        print("StatisticsError:", e)

In [ ]:
#the ordering of the means
print("[2, 8] ->", st.mean([2, 8]), st.geometric_mean([2, 8]), st.harmonic_mean([2, 8]))   # 5, 4, 3.2

trial = rng.uniform(1, 100, size=1000)
am, gm, hm = trial.mean(), stats.gmean(trial), stats.hmean(trial)
print(f"AM {am:.2f} >= GM {gm:.2f} >= HM {hm:.2f}:", am >= gm >= hm)

same = [7, 7, 7]
print(st.mean(same), st.geometric_mean(same), st.harmonic_mean(same))     # all equal

In [ ]:
#the salary example and a robust alternative
salaries = np.array([30, 32, 35, 36, 38, 40, 42, 500])      # thousands, with one executive
print("mean              :", salaries.mean())               # 94.125: higher than 7 of 8 people
print("median            :", np.median(salaries))           # 37.0
print("trimmed mean      :", round(stats.trim_mean(salaries, 0.125), 3))   # drops 1 value from each end
print("mean without 500  :", round(salaries[:-1].mean(), 3))

In [ ]:
#skewed shapes
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
datasets = {
    "Right-skewed": rng.lognormal(3, 0.6, 5000),
    "Symmetric": rng.normal(30, 6, 5000),
    "Left-skewed": 100 - rng.lognormal(3, 0.6, 5000),
}
for ax, (title, data) in zip(axes, datasets.items()):
    ax.hist(data, bins=50, color="lightsteelblue", edgecolor="white")
    ax.axvline(data.mean(), color="tomato", lw=2, label=f"mean {data.mean():.1f}")
    ax.axvline(np.median(data), color="seagreen", lw=2, ls="--", label=f"median {np.median(data):.1f}")
    ax.set_title(f"{title} (skew {stats.skew(data):+.2f})")
    ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
#a quick summary in pandas
s = pd.Series(salaries, name="salary_k")
print(s.describe())                                  # count, mean, std, min, quartiles (Day S2), max
print("mean", s.mean(), "median", s.median(), "mode", s.mode().tolist())